In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

def initialize_plotting():
    import matplotlib as mpl
    %config InlineBackend.figure_format = 'svg'
    label_size = 20
    mpl.rcParams['xtick.labelsize'] = label_size
    mpl.rcParams['ytick.labelsize'] = label_size
    mpl.rcParams['legend.fontsize'] = 14
    plt.rc('font', family='serif')
    mpl.rcParams.update({'font.size': 16})
    mpl.rcParams['text.usetex'] = False
    mpl.rcParams['figure.dpi'] = 120
initialize_plotting()

In [ ]:
#inciso b

# constantes físicas
m_e_c2 = 0.510998      # Masa del electrón en MeV
m_p_c2 = 938.272       # Masa del protón en MeV
K = 0.307075           # Constante K en MeV cm^2 / mol
Z_A = 10.0 / 18.015    # Z/A para agua 
I_eV = 75.0            # Potencial de excitación en eV
I = I_eV * 1e-6        # Conversión de I a MeV
rho = 1.0              # Densidad del agua en g/cm^3

def bethe_bloch(E):
    if E <= 0:
        return 0.0
    
    # Cálculos relativistas
    gamma = (E + m_p_c2) / m_p_c2
    beta2 = 1.0 - (1.0 / gamma**2)
    
    if beta2 <= 0:
        return 0.0
        
    ratio_m = m_e_c2 / m_p_c2
    T_max = (2 * m_e_c2 * beta2 * gamma**2) / (1 + 2 * gamma * ratio_m + ratio_m**2)
    
    termino_log = 0.5 * np.log((2 * m_e_c2 * beta2 * gamma**2 * T_max) / (I**2))
    
    dEdx = K * Z_A * (1.0 / beta2) * (termino_log - beta2) * rho
    return dEdx

def calcular_rango_csda(E0):
    #integramos desde un valor pequeño  en vez de 0 
    rango, _ = quad(lambda E: 1.0 / bethe_bloch(E), 0.1, E0, limit=1000)
    return rango

#energías solicitadas a
energias = [50, 150, 250]

for E0 in energias:
    rango = calcular_rango_csda(E0)
    print(f"Energía inicial E0 = {E0:3d} MeV  -->  Rango CSDA = {rango:6.3f} cm")

In [ ]:
#inciso c

# configuración de la simulación
N_protones = 10000
E0 = 150.0       # MeV
dx = 0.01        # cm 
z_max = 20.0     # cm 

# creación fantoma de agua 
n_bins = int(z_max / dx)
z_array = np.linspace(0, z_max, n_bins)
dosis_c = np.zeros(n_bins)  # Aquí acumularemos la energía depositada

for i in range(N_protones):
    E_actual = E0
    
    for j in range(n_bins):
        # condición: si el protón ya casi no tiene energía se detiene
        if E_actual <= 0.1:
            dosis_c[j] += E_actual
            break
            
        # Calcular poder de frenado y energía perdida en este diferencial dx
        dEdx = bethe_bloch(E_actual)
        dE = dEdx * dx
        
        # Evitar depositar más energía de la que el protón posee
        if dE > E_actual:
            dE = E_actual
            
        # Acumular la dosis en la posición z actual
        dosis_c[j] += dE
        
        # Actualizar la energía restante del protón
        E_actual -= dE

indice_pico = np.argmax(dosis_c)
posicion_pico = z_array[indice_pico]

# graficamos
plt.figure(figsize=(10, 6))
plt.plot(z_array, dosis_c, color='blue', linewidth=2, label='D(z) - Sin straggling')
plt.axvline(x=posicion_pico, color='red', linestyle='--', 
            label=f'Pico de Bragg en z = {posicion_pico:.2f} cm')

plt.xlabel('Profundidad z (cm)')
plt.ylabel('Energía Total Depositada (MeV)')
plt.legend()
plt.grid(True, alpha=0.5)
plt.xlim(0, 18) # Acotar el eje x para ver mejor la curva
plt.show()

In [ ]:
# inciso d

dosis_straggling = np.zeros(n_bins)
posiciones_parada = [] 

for i in range(N_protones):
    E_actual = E0
    
    for j in range(n_bins):
        if E_actual <= 0.1:
            dosis_straggling[j] += E_actual
            posiciones_parada.append(z_array[j])
            break
            
        # (<Delta E>)
        dEdx = bethe_bloch(E_actual)
        mean_dE = dEdx * dx
        
        # varianza (Bohr straggling)
        gamma = (E_actual + m_p_c2) / m_p_c2
        beta2 = 1.0 - (1.0 / gamma**2)
        
        if beta2 > 0:
            var_E = K * m_e_c2 * Z_A * rho * (1.0 / beta2) * dx
        else:
            var_E = 0.0
            
        # muestreo Monte Carlo 
        sigma_E = np.sqrt(var_E)
        # delta E 
        dE = np.random.normal(mean_dE, sigma_E)
        
        # Correcciones físicas
        if dE < 0:
            dE = 0 
        if dE > E_actual:
            dE = E_actual
            
        dosis_straggling[j] += dE
        E_actual -= dE

# ensanchamiento espacial del pico
sigma_R = np.std(posiciones_parada)

# comparacion
plt.figure(figsize=(10, 6))

plt.plot(z_array, dosis_c, color='gray', linestyle='--', label='D(z) - Sin straggling')

plt.plot(z_array, dosis_straggling, color='blue', linewidth=2, label='D(z) - Con straggling (Monte Carlo)')

plt.xlabel('Profundidad z (cm)')
plt.ylabel('Energía Total Depositada (MeV)')
plt.legend()
plt.grid(True, alpha=0.5)

plt.savefig("grafico_pico_bragg.pdf", format="pdf", bbox_inches="tight")
 
plt.show()

print(f"Ensanchamiento del pico (sigma_R): {sigma_R:.4f} cm")